# Phases 3 + 4 + 5 — Swin-Tiny on Food-101
Unified driver: split → train → evaluate → analyze.

**Environment-aware:** detects Kaggle vs local at section 0.

## 0. Environment + paths

In [ ]:
from pathlib import Path
from foodnet.utils.paths import detect_env
ENV = detect_env()
if ENV == 'kaggle':
    IMAGES_ROOT = Path('/kaggle/input/food-101/food-101/images')
    OUTPUT_DIR = Path('/kaggle/working/runs/baseline')
    SPLITS_DIR = Path('/kaggle/working/splits')
else:
    IMAGES_ROOT = Path('data/food-101/images')
    OUTPUT_DIR = Path('runs/baseline')
    SPLITS_DIR = Path('splits')
print({'env': ENV, 'images_root': str(IMAGES_ROOT), 'output_dir': str(OUTPUT_DIR), 'splits_dir': str(SPLITS_DIR)})

## 1. Generate (or refresh) 8:1:1 splits

In [ ]:
REGENERATE_SPLITS = False  # set True to overwrite committed splits
if REGENERATE_SPLITS or not (SPLITS_DIR / 'train.csv').exists():
    !foodnet-split --images-root "$IMAGES_ROOT" --output-dir "$SPLITS_DIR" --seed 42 --ratios 0.8,0.1,0.1
else:
    print('splits already present at', SPLITS_DIR)

## 2. Train Swin-Tiny baseline

In [ ]:
import os
EPOCHS = 30
BATCH = 64 if ENV == 'kaggle' else 32
WANDB = '--wandb' if os.environ.get('WANDB_API_KEY') else '--no-wandb'
!foodnet-train \
  --splits-dir "$SPLITS_DIR" \
  --images-root "$IMAGES_ROOT" \
  --output-dir "$OUTPUT_DIR" \
  --num-classes 101 --epochs $EPOCHS --batch-size $BATCH \
  --lr 5e-4 --weight-decay 0.05 --layer-decay 0.75 \
  --warmup-epochs 2 --grad-clip 1.0 --label-smoothing 0.1 \
  --mixup-alpha 0.8 --cutmix-alpha 1.0 --re-prob 0.25 \
  --early-stop --early-stop-patience 8 \
  --seed 42 --device cuda --amp $WANDB --wandb-project foodnet

## 3. Evaluate on test split + analyze

In [ ]:
PRED_PARQUET = OUTPUT_DIR / 'test_preds.parquet'
DASH_DIR = OUTPUT_DIR / 'analysis'
!foodnet-evaluate \
  --checkpoint "$OUTPUT_DIR/best.pt" \
  --splits-dir "$SPLITS_DIR" --images-root "$IMAGES_ROOT" \
  --split test --output-parquet "$PRED_PARQUET" \
  --num-classes 101 --batch-size 128 --device cuda --amp
!foodnet-analyze \
  --predictions "$PRED_PARQUET" \
  --classes "$SPLITS_DIR/classes.txt" \
  --output-dir "$DASH_DIR"

## 4. Render dashboard summary

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown((DASH_DIR / 'dashboard.md').read_text()))